In [2]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

import sklearn
print("scikit-learn:", sklearn.__version__)

scikit-learn: 1.8.0


## Revisión de Dataset EMPRESAS.

### Verificamos datos básicos:

In [3]:
BASE = Path(r"C:\IA_Investigacion\Deteccion_Corrupcion")

DATA_PROC = BASE / "data" / "processed"

print("BASE:", BASE)
print("DATA_PROC:", DATA_PROC)

BASE: C:\IA_Investigacion\Deteccion_Corrupcion
DATA_PROC: C:\IA_Investigacion\Deteccion_Corrupcion\data\processed


In [5]:
import pandas as pd

df = pd.read_parquet(DATA_PROC / "dataset_empresa_v3_4niveles_model.parquet")

print("Shape:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())

print("\nTipos:")
print(df.dtypes)

print("\nPrimeras filas:")
display(df.head())

print("\nDistribución target:")
print(df["y_riesgo_empresa_4niveles"].value_counts())

Shape: (8047, 26)

Columnas:
['CODIGO_RUC', 'NOMBRE_EMPRESA_ROL', 'RUC_CONTRAPARTE', 'NOMBRE_CONTRAPARTE', 'rol_empresa', 'n_obras_asociadas_1a', 'n_registros_1a', 'es_consorcio', 'capacidad_maxima_contratacion', 'capacidad_libre_contratacion', 'estado_empresa', 'condicion_empresa', 'n_obras_representante', 'n_representantes_legales', 'representante_legal_principal', 'NOMBRE_EMPRESA', 'y_riesgo_empresa_4niveles', 'n_vinculos_total', 'n_contrapartes_unicas', 'n_veces_ganador', 'n_veces_participante', 'ratio_ganador', 'ratio_participante', 'n_registros_hora_oferta', 'n_horas_oferta_distintas', 'hora_oferta_moda']

Tipos:
CODIGO_RUC                        object
NOMBRE_EMPRESA_ROL                object
RUC_CONTRAPARTE                   object
NOMBRE_CONTRAPARTE                object
rol_empresa                       object
n_obras_asociadas_1a             float64
n_registros_1a                   float64
es_consorcio                     float64
capacidad_maxima_contratacion    float64
capa

,CODIGO_RUC,NOMBRE_EMPRESA_ROL,RUC_CONTRAPARTE,NOMBRE_CONTRAPARTE,rol_empresa,n_obras_asociadas_1a,n_registros_1a,es_consorcio,capacidad_maxima_contratacion,capacidad_libre_contratacion,...,y_riesgo_empresa_4niveles,n_vinculos_total,n_contrapartes_unicas,n_veces_ganador,n_veces_participante,ratio_ganador,ratio_participante,n_registros_hora_oferta,n_horas_oferta_distintas,hora_oferta_moda
0,20100913225,JNR CONSULTORES SOCIEDAD ANONIMA,10086051546,CONSORCIO SUPERVISOR AREQUIPA-LAJOYA,ganador,1.0,1.0,0.0,NaN,0.0,...,2,15.0,13.0,13.0,2.0,0.866667,0.133333,1.0,1.0,10:00
1,20100913225,JNR CONSULTORES SOCIEDAD ANONIMA,20100153671,LAGESA INGENIEROS CONSULTORES S.A.(*),ganador,1.0,1.0,0.0,NaN,0.0,...,2,15.0,13.0,13.0,2.0,0.866667,0.133333,1.0,1.0,10:00
2,20100913225,JNR CONSULTORES SOCIEDAD ANONIMA,20100913225,JNR CONSULTORES SOCIEDAD ANONIMA,ganador,1.0,1.0,0.0,NaN,0.0,...,2,15.0,13.0,13.0,2.0,0.866667,0.133333,1.0,1.0,10:00
3,20100913225,JNR CONSULTORES SOCIEDAD ANONIMA,20112860381,CONSORCIO SUPERVISOR AREQUIPA,ganador,1.0,1.0,0.0,NaN,0.0,...,2,15.0,13.0,13.0,2.0,0.866667,0.133333,1.0,1.0,10:00
4,20100913225,JNR CONSULTORES SOCIEDAD ANONIMA,20137114705,SERVICIO DE CONSULTORES ANDINOS SOCIEDAD ANONIMA,ganador,1.0,1.0,0.0,NaN,0.0,...,2,15.0,13.0,13.0,2.0,0.866667,0.133333,1.0,1.0,10:00



Distribución target:
y_riesgo_empresa_4niveles
0    3336
2    3049
3    1122
1     540
Name: count, dtype: int64


## Revisión de Datasets Funcionario

### Datos básicos

In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

# ===============================
# CONFIGURACIÓN
# ===============================

BASE = Path(r"C:\IA_Investigacion\Deteccion_Corrupcion")
DATA_PROC = BASE / "data" / "processed"

print("BASE:", BASE)
print("DATA_PROC:", DATA_PROC)

# ===============================
# ARCHIVOS A REVISAR
# ===============================

files = {
    "funcionario_obra": DATA_PROC / "dataset_funcionario_v3_4niveles_obra.parquet",
    "funcionario_unico": DATA_PROC / "dataset_funcionario_v3_4niveles_unico.parquet",
    "funcionario_vinculo": DATA_PROC / "dataset_funcionario_v3_4niveles_vinculo.parquet",
    "funcionario_empresa": DATA_PROC / "dataset_funcionario_v3_4niveles_empresa.parquet",
    "funcionario_model": DATA_PROC / "dataset_funcionario_v3_4niveles_model.parquet",
}

# ===============================
# FUNCIONES DE APOYO
# ===============================

def find_cols(df, patterns):
    cols = []
    for c in df.columns:
        cu = c.upper()
        if any(p.upper() in cu for p in patterns):
            cols.append(c)
    return cols


def safe_nunique(df, col):
    return df[col].nunique(dropna=True) if col in df.columns else None


def pct_null(df, col):
    return round(df[col].isna().mean() * 100, 2) if col in df.columns else None


def print_block(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


# ===============================
# CARGA Y REVISIÓN GENERAL
# ===============================

datasets = {}
summary_rows = []

for name, path in files.items():
    print_block(f"REVISANDO: {name}")

    if not path.exists():
        print("NO EXISTE:", path)
        continue

    df = pd.read_parquet(path)
    datasets[name] = df

    print("Archivo:", path.name)
    print("Shape:", df.shape)

    dni_cols = find_cols(df, ["DNI", "DOCUMENTO", "DOC_IDENTIDAD", "FUNCIONARIO"])
    name_cols = find_cols(df, ["NOMBRE", "APELLIDO"])
    obra_cols = find_cols(df, ["IDENTIFICADOR_OBRA", "CODIGO_OBRA", "CODIGO_UNICO", "COD_UNICO"])
    empresa_cols = find_cols(df, ["RUC", "EMPRESA"])
    riesgo_cols = find_cols(df, ["RIESGO", "TARGET", "Y_"])
    id_cols = find_cols(df, ["ID_", "IDENTIFICADOR", "CODIGO"])

    print("\nColumnas DNI / funcionario:")
    print(dni_cols)

    print("\nColumnas nombres:")
    print(name_cols)

    print("\nColumnas obra:")
    print(obra_cols)

    print("\nColumnas empresa:")
    print(empresa_cols)

    print("\nColumnas riesgo / target:")
    print(riesgo_cols)

    print("\nColumnas ID / códigos:")
    print(id_cols)

    print("\nTipos de datos:")
    display(df.dtypes.to_frame("dtype"))

    print("\nPrimeras filas:")
    display(df.head())

    # Distribución de targets si existen
    for col in riesgo_cols:
        if df[col].nunique(dropna=True) <= 20:
            print(f"\nDistribución de {col}:")
            display(df[col].value_counts(dropna=False).sort_index().to_frame("cantidad"))

    # Resumen de nulos top
    print("\nTop 15 columnas con más nulos:")
    display(
        df.isna()
        .mean()
        .sort_values(ascending=False)
        .head(15)
        .mul(100)
        .round(2)
        .to_frame("% nulos")
    )

    summary_rows.append({
        "dataset": name,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "dni_cols": ", ".join(dni_cols),
        "name_cols": ", ".join(name_cols),
        "obra_cols": ", ".join(obra_cols),
        "empresa_cols": ", ".join(empresa_cols),
        "riesgo_cols": ", ".join(riesgo_cols),
        "id_cols": ", ".join(id_cols),
        "tiene_IDENTIFICADOR_OBRA": "IDENTIFICADOR_OBRA" in df.columns,
        "tiene_CODIGO_OBRA": "CODIGO_OBRA" in df.columns,
        "n_IDENTIFICADOR_OBRA": safe_nunique(df, "IDENTIFICADOR_OBRA"),
        "n_CODIGO_OBRA": safe_nunique(df, "CODIGO_OBRA"),
    })

summary = pd.DataFrame(summary_rows)

print_block("RESUMEN GENERAL DE DATASETS FUNCIONARIO")
display(summary)

# ===============================
# PREGUNTAS CLAVE PARA DECIDIR V4
# ===============================

print_block("PREGUNTAS CLAVE")

for name, df in datasets.items():
    print(f"\nDATASET: {name}")

    # 1. Tiene vínculo con obra
    obra_cols = find_cols(df, ["IDENTIFICADOR_OBRA", "CODIGO_OBRA", "CODIGO_UNICO", "COD_UNICO"])
    if obra_cols:
        print("1. ¿Tiene vínculo con obra?: SÍ")
        for col in obra_cols:
            print(f"   - {col}: únicos={df[col].nunique(dropna=True)}, nulos={pct_null(df, col)}%")
    else:
        print("1. ¿Tiene vínculo con obra?: NO")

    # 2. Tiene identificadores personales
    dni_cols = find_cols(df, ["DNI", "DOCUMENTO", "DOC_IDENTIDAD"])
    if dni_cols:
        print("2. ¿Tiene DNI/documentos?: SÍ -> solo trazabilidad, NO modelo")
        print("   ", dni_cols)
    else:
        print("2. ¿Tiene DNI/documentos?: NO")

    # 3. Tiene nombres
    name_cols = find_cols(df, ["NOMBRE", "APELLIDO"])
    if name_cols:
        print("3. ¿Tiene nombres?: SÍ -> solo auditoría, NO modelo")
        print("   ", name_cols)
    else:
        print("3. ¿Tiene nombres?: NO")

    # 4. Tiene target/proxy
    riesgo_cols = find_cols(df, ["RIESGO", "TARGET", "Y_"])
    if riesgo_cols:
        print("4. ¿Tiene señales de riesgo?: SÍ")
        print("   ", riesgo_cols)
    else:
        print("4. ¿Tiene señales de riesgo?: NO")

    # 5. Es candidato para maestro
    if any(c in df.columns for c in ["IDENTIFICADOR_OBRA", "CODIGO_OBRA", "CODIGO_UNICO", "COD_UNICO"]):
        print("5. ¿Puede alimentar dataset maestro?: SÍ")
    else:
        print("5. ¿Puede alimentar dataset maestro?: NO directamente")

# ===============================
# DETECCIÓN DE COLUMNAS PROHIBIDAS PARA MODELO
# ===============================

print_block("COLUMNAS QUE NO DEBEN ENTRAR AL MODELO FINAL")

for name, df in datasets.items():
    forbidden_patterns = [
        "DNI",
        "DOCUMENTO",
        "DOC_IDENTIDAD",
        "NOMBRE",
        "APELLIDO",
        "RUC",
        "CODIGO",
        "IDENTIFICADOR",
        "ID_",
    ]

    bad_cols = find_cols(df, forbidden_patterns)

    print(f"\n{name}:")
    print(bad_cols if bad_cols else "Sin columnas prohibidas aparentes")

# ===============================
# PROPUESTA DE FEATURES POR OBRA
# ===============================

print_block("PROPUESTA AUTOMÁTICA DE DATASET FUNCIONARIO FEATURES POR OBRA")

# Buscar el mejor dataset con vínculo a obra y funcionario
priority = [
    "funcionario_obra",
    "funcionario_vinculo",
    "funcionario_model",
    "funcionario_empresa",
    "funcionario_unico",
]

selected_name = None
selected_df = None

for name in priority:
    if name in datasets:
        df = datasets[name]
        obra_cols = find_cols(df, ["IDENTIFICADOR_OBRA", "CODIGO_OBRA", "CODIGO_UNICO", "COD_UNICO"])
        dni_cols = find_cols(df, ["DNI", "DOCUMENTO", "DOC_IDENTIDAD"])
        if obra_cols and dni_cols:
            selected_name = name
            selected_df = df.copy()
            break

if selected_df is None:
    print("No se encontró un dataset con columna de obra y DNI/documento.")
    print("Necesitamos revisar manualmente las columnas.")
else:
    print("Dataset seleccionado para construir features por obra:", selected_name)

    obra_col = find_cols(selected_df, ["IDENTIFICADOR_OBRA", "CODIGO_OBRA", "CODIGO_UNICO", "COD_UNICO"])[0]
    dni_col = find_cols(selected_df, ["DNI", "DOCUMENTO", "DOC_IDENTIDAD"])[0]

    print("Columna obra seleccionada:", obra_col)
    print("Columna funcionario seleccionada:", dni_col)

    df_feat = selected_df.copy()

    # Buscar columnas de riesgo
    riesgo_cols = find_cols(df_feat, ["y_riesgo_funcionario", "RIESGO_FUNCIONARIO", "riesgo_func", "RIESGO", "Y_"])

    # Elegir target/proxy si existe
    riesgo_col = None
    for c in riesgo_cols:
        if df_feat[c].nunique(dropna=True) <= 10:
            riesgo_col = c
            break

    print("Columna de riesgo/proxy seleccionada:", riesgo_col)

    # Normalizar
    df_feat[obra_col] = df_feat[obra_col].astype(str).str.strip()
    df_feat[dni_col] = df_feat[dni_col].astype(str).str.strip()

    # Crear features básicas
    agg_dict = {
        "func_n_funcionarios": (dni_col, "nunique"),
        "func_n_registros": (dni_col, "size"),
    }

    # Roles si hay columna rol/cargo
    rol_cols = find_cols(df_feat, ["ROL", "CARGO", "PUESTO", "FUNCION"])
    if rol_cols:
        rol_col = rol_cols[0]
        agg_dict["func_n_roles_distintos"] = (rol_col, "nunique")
        print("Columna rol/cargo seleccionada:", rol_col)
    else:
        print("No se encontró columna rol/cargo.")

    # Riesgo funcionario si existe
    if riesgo_col:
        df_feat[riesgo_col] = pd.to_numeric(df_feat[riesgo_col], errors="coerce")

        func_por_obra = df_feat.groupby(obra_col).agg(
            **agg_dict,
            func_riesgo_max=(riesgo_col, "max"),
            func_riesgo_mean=(riesgo_col, "mean"),
            func_riesgo_std=(riesgo_col, "std"),
            func_pct_alto_riesgo=(riesgo_col, lambda s: (s >= 3).mean()),
        ).reset_index()

        func_por_obra["func_riesgo_std"] = func_por_obra["func_riesgo_std"].fillna(0)
    else:
        func_por_obra = df_feat.groupby(obra_col).agg(**agg_dict).reset_index()

    # Renombrar llave a CODIGO_OBRA temporal si no se llama así
    if obra_col != "CODIGO_OBRA":
        func_por_obra = func_por_obra.rename(columns={obra_col: "CODIGO_OBRA"})

    print("\nFeatures por obra generadas preliminarmente:")
    print(func_por_obra.shape)
    display(func_por_obra.head())

    print("\nColumnas generadas:")
    print(func_por_obra.columns.tolist())

# ===============================
# RECOMENDACIÓN FINAL
# ===============================

print_block("RECOMENDACIÓN FINAL")

print("""
1. El dataset de funcionario NO debe entrenarse directamente.
2. Debe generar features agregadas por obra para el dataset maestro.
3. DNI, nombres, documentos y códigos se usan solo para trazabilidad/joins.
4. La salida ideal del notebook V4 debe ser:

   data/processed/dataset_funcionario_v4_features_por_obra.parquet

5. Esta salida debe contener:
   - CODIGO_OBRA o IDENTIFICADOR_OBRA como llave temporal
   - func_n_funcionarios
   - func_n_registros
   - func_n_roles_distintos
   - func_riesgo_max
   - func_riesgo_mean
   - func_pct_alto_riesgo

6. El dataset maestro final debe excluir la llave como feature antes de entrenar.
""")

BASE: C:\IA_Investigacion\Deteccion_Corrupcion
DATA_PROC: C:\IA_Investigacion\Deteccion_Corrupcion\data\processed

REVISANDO: funcionario_obra
Archivo: dataset_funcionario_v3_4niveles_obra.parquet
Shape: (1613, 16)

Columnas DNI / funcionario:
['CODIGO_DNI', 'TIPO_DOCUMENTO', 'score_riesgo_funcionario', 'y_riesgo_funcionario_4niveles']

Columnas nombres:
['NOMBRE_COMPLETO']

Columnas obra:
['CODIGO_OBRA']

Columnas empresa:
[]

Columnas riesgo / target:
['score_riesgo_funcionario', 'y_riesgo_funcionario_4niveles']

Columnas ID / códigos:
['CODIGO_OBRA', 'CODIGO_DNI']

Tipos de datos:


,dtype
CODIGO_OBRA,object
CODIGO_DNI,object
TIPO_DOCUMENTO,object
NOMBRE_COMPLETO,object
RESP_PENAL,int64
RESP_ADMIN,int64
RESP_CIVIL,int64
SANCIONADA_SERVIR,int64
SANCIONADA_RNP,int64
INHABILITADOS_SERVIR,int64



Primeras filas:


,CODIGO_OBRA,CODIGO_DNI,TIPO_DOCUMENTO,NOMBRE_COMPLETO,RESP_PENAL,RESP_ADMIN,RESP_CIVIL,SANCIONADA_SERVIR,SANCIONADA_RNP,INHABILITADOS_SERVIR,score_responsabilidad,score_sancion,score_riesgo_funcionario,y_riesgo_funcionario_4niveles,n_obras_vinculadas,n_registros_obra
0,19777,21461664,DNI,LUZ YOLANDA ESQUIVEL CERON,0,0,0,0,0,0,0,0,0,0,2,2
1,19777,32977736,DNI,Hugo Antonio Cotos Perez,0,4,4,0,0,0,16,0,16,3,2,2
2,19777,41798624,DNI,FREDDY MANCHA CASO,0,2,0,0,0,0,4,0,4,2,2,2
3,19777,43584310,DNI,MARIA CECILIA PIÑAN INDACOCHEA,0,0,0,0,0,0,0,0,0,0,6,6
4,19777,9382177,DNI,ELKA CAROLINA ROSALES LAURENTE,0,1,2,0,0,0,6,0,6,2,3,3



Distribución de y_riesgo_funcionario_4niveles:


...


RESUMEN GENERAL DE DATASETS FUNCIONARIO


,dataset,filas,columnas,dni_cols,name_cols,obra_cols,empresa_cols,riesgo_cols,id_cols,tiene_IDENTIFICADOR_OBRA,tiene_CODIGO_OBRA,n_IDENTIFICADOR_OBRA,n_CODIGO_OBRA
0,funcionario_obra,1613,16,"CODIGO_DNI, TIPO_DOCUMENTO, score_riesgo_funci...",NOMBRE_COMPLETO,CODIGO_OBRA,,"score_riesgo_funcionario, y_riesgo_funcionario...","CODIGO_OBRA, CODIGO_DNI",False,True,None,178.0
1,funcionario_unico,739,15,"CODIGO_DNI, TIPO_DOCUMENTO, score_riesgo_funci...",NOMBRE_COMPLETO,,,"score_riesgo_funcionario, y_riesgo_funcionario...",CODIGO_DNI,False,False,None,NaN
2,funcionario_vinculo,4862,23,"CODIGO_DNI, TIPO_DOCUMENTO, score_riesgo_funci...","NOMBRE_COMPLETO, NOMBRE_EMPRESA",CODIGO_OBRA,"NOMBRE_EMPRESA, RUC_EMPRESA, tipo_vinculo_empr...","score_riesgo_funcionario, y_riesgo_funcionario...","CODIGO_DNI, CODIGO_OBRA",False,True,None,178.0
3,funcionario_empresa,3249,14,"CODIGO_DNI, NOMBRE_FUNCIONARIO, score_riesgo_f...","NOMBRE_EMPRESA, NOMBRE_FUNCIONARIO",,"RUC_EMPRESA, NOMBRE_EMPRESA, tipo_vinculo_empr...","score_riesgo_funcionario, y_riesgo_funcionario...",CODIGO_DNI,False,False,None,NaN
4,funcionario_model,4219,13,"CODIGO_DNI, y_riesgo_funcionario_4niveles, COD...","NOMBRE_COMPLETO, NOMBRE_EMPRESA",CODIGO_OBRA,"NOMBRE_EMPRESA, RUC_EMPRESA, n_empresas_vincul...",y_riesgo_funcionario_4niveles,"CODIGO_DNI, CODIGO_OBRA, CODIGO_DNI_GROUP",False,True,None,178.0



PREGUNTAS CLAVE

DATASET: funcionario_obra
1. ¿Tiene vínculo con obra?: SÍ
   - CODIGO_OBRA: únicos=178, nulos=0.0%
2. ¿Tiene DNI/documentos?: SÍ -> solo trazabilidad, NO modelo
    ['CODIGO_DNI', 'TIPO_DOCUMENTO']
3. ¿Tiene nombres?: SÍ -> solo auditoría, NO modelo
    ['NOMBRE_COMPLETO']
4. ¿Tiene señales de riesgo?: SÍ
    ['score_riesgo_funcionario', 'y_riesgo_funcionario_4niveles']
5. ¿Puede alimentar dataset maestro?: SÍ

DATASET: funcionario_unico
1. ¿Tiene vínculo con obra?: NO
2. ¿Tiene DNI/documentos?: SÍ -> solo trazabilidad, NO modelo
    ['CODIGO_DNI', 'TIPO_DOCUMENTO']
3. ¿Tiene nombres?: SÍ -> solo auditoría, NO modelo
    ['NOMBRE_COMPLETO']
4. ¿Tiene señales de riesgo?: SÍ
    ['score_riesgo_funcionario', 'y_riesgo_funcionario_4niveles']
5. ¿Puede alimentar dataset maestro?: NO directamente

DATASET: funcionario_vinculo
1. ¿Tiene vínculo con obra?: SÍ
   - CODIGO_OBRA: únicos=178, nulos=66.82%
2. ¿Tiene DNI/documentos?: SÍ -> solo trazabilidad, NO modelo
    ['CODIGO_

,CODIGO_OBRA,func_n_funcionarios,func_n_registros,func_n_roles_distintos,func_riesgo_max,func_riesgo_mean,func_riesgo_std,func_pct_alto_riesgo
0,102021,6,6,3,2,0.500000,0.836660,0.000000
1,102022,7,7,3,3,0.857143,1.463850,0.285714
2,102591,6,6,3,3,1.000000,1.549193,0.333333
3,105653,8,8,1,0,0.000000,0.000000,0.000000
4,106748,8,8,2,3,0.375000,1.060660,0.125000



Columnas generadas:
['CODIGO_OBRA', 'func_n_funcionarios', 'func_n_registros', 'func_n_roles_distintos', 'func_riesgo_max', 'func_riesgo_mean', 'func_riesgo_std', 'func_pct_alto_riesgo']

RECOMENDACIÓN FINAL

1. El dataset de funcionario NO debe entrenarse directamente.
2. Debe generar features agregadas por obra para el dataset maestro.
3. DNI, nombres, documentos y códigos se usan solo para trazabilidad/joins.
4. La salida ideal del notebook V4 debe ser:

   data/processed/dataset_funcionario_v4_features_por_obra.parquet

5. Esta salida debe contener:
   - CODIGO_OBRA o IDENTIFICADOR_OBRA como llave temporal
   - func_n_funcionarios
   - func_n_registros
   - func_n_roles_distintos
   - func_riesgo_max
   - func_riesgo_mean
   - func_pct_alto_riesgo

6. El dataset maestro final debe excluir la llave como feature antes de entrenar.

